In [1]:
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from pathlib import Path
from dotenv import load_dotenv

C:\Users\Shagufta\AppData\Local\Temp\ipykernel_34016\1762874745.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader, DirectoryLoader


In [2]:
# load the api keys
load_dotenv()

True

In [3]:
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

In [4]:
# create the llm 
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

In [5]:
response = llm.invoke("What is the capital of India?")
print(response.text)

The capital of India is **New Delhi**.


In [6]:
# create embedding model
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


d:\Data_Science_N_AI\LLMOPS\LLMOPS_RAG_APP\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6176.03it/s]


In [7]:
vector = embeddings.embed_query("What is machine learning?")

print("Successfully generated embedding")
print("Dimensions:", len(vector))

Successfully generated embedding
Dimensions: 384


In [8]:
# load the docs from source

loader = DirectoryLoader(path = Path("./data/processed/").as_posix(),
                        loader_cls = TextLoader,
                        show_progress=True)

docs = loader.load()
                        

100%|██████████| 14/14 [00:00<00:00, 1836.96it/s]


In [9]:
docs[0].page_content[0:100]

'Pakistan, Ahmedabad, Patna, Dheradoon, Subodh, where are you from? I am from Dheradoon too.\nYou are '

In [10]:
# chunk the text

chunker  = RecursiveCharacterTextSplitter(chunk_size = 500,
                                          chunk_overlap = 50)
chunks = chunker.split_documents(docs)

In [11]:
print(f"chunks: {len(chunks)}")

chunks: 2617


In [12]:
# vector store
vs = Chroma(collection_name="rag_demo",
            embedding_function=embeddings,
            persist_directory=Path("./saved_embedings").as_posix())

vs.add_documents(chunks)

['f3ebc8b5-91ba-4112-a638-4c2b001c30dd',
 '980e863a-8087-4c50-805e-27f101be1321',
 'b18c0910-0d95-4e35-90f1-06a06afeb1de',
 'f64a0baa-5a9a-41c3-8c1c-05b91724784a',
 '732ae85d-51b8-4c88-a577-f4c9100a4421',
 '959b565d-55a0-4bd5-94ab-bf56f803b21c',
 '40ae1606-13be-45e2-8b7c-390dbdde55e2',
 '49a9c89d-d34d-4d2f-8346-a240ac82aca6',
 '3d53b318-efbb-4150-9f0e-a1258fc641d4',
 'f8a9e662-43a6-4f67-b5f8-e76ceac1a40b',
 '6adc2904-4f32-4ed4-bb3d-a3f42b148ea9',
 '37f2dabb-b2ec-4253-bd87-f51f1e96da34',
 '2cd87dce-9c3a-4c36-a418-2d57d9073d45',
 '5285f335-e067-4e47-ab7b-491bf7149762',
 '7662c66f-2907-456f-8d52-1886d39178a1',
 '8ebf3ed2-3750-404a-83eb-fdb6e9d55161',
 '71d883d5-205c-4b0f-8bd6-4aae16b690f0',
 '4dd52de4-aac3-46af-b144-dd87f47fd6b6',
 '3bb29a9d-b30f-47d5-aed5-feef53e5f08f',
 '35a66b95-f4fe-4be9-b1bc-78a2471a1a0b',
 '13722a57-4a8e-4250-bae4-29cb73cb25c9',
 '24eacf9f-d41a-4119-ba7f-f3a0ebdb7b8f',
 'fe7b6358-7033-4cf7-982e-d47ae3656d9c',
 'b748089d-ad17-4ea8-b4d9-aad86b7c3cd1',
 '65786356-fd64-

In [13]:
# create the retriever

retriever = vs.as_retriever(search_kwargs={'k':3})

### Langgraph Workflow

In [14]:
# langgraph imports
from langgraph.graph import StateGraph, START, END
from typing import Literal, TypedDict

In [15]:
class RAGState(TypedDict):
    query : str
    retrieved_docs : list[Document]
    context : str
    prompt : ChatPromptTemplate
    response : str
    

In [16]:
def retrieve(state : RAGState) -> dict:
    query = state['query']
    retrieved_docs = retriever.invoke(query)

    context = "\n\n".join([doc.page_content for doc in retrieved_docs])
    return {"retrieved_context" : retrieved_docs,
            "context" : context}

In [17]:
def augmentation(state: RAGState) -> dict:
    prompt = ChatPromptTemplate.from_messages([
        ("System", """You are a helpful assistant. Answer the user query
                        based on the given context only. If you do not know the answer
                        say I don't know. Do not add any preamble to the response"""),
        ("human", 'context: {context}\n\nquery: {query}')
    ])

    return {"prompt" : prompt}

In [18]:
from langchain_core.output_parsers import StrOutputParser

In [19]:
def generation(state: RAGState) -> dict:

    query = state['query']
    context = state['context']
    prompt = state['prompt']

    rag_chain = prompt | llm | StrOutputParser()
    response = rag_chain.invoke({'context' : context, 'query': query})

    return {'response' : response}

In [ ]:
# create the graph
graph_builder = StateGraph(RAGState)

graph_builder.add_node('retrieve', retrieve)
graph_builder.add_node('augmentation', augmentation)
graph_builder.add_node('generation', generation)

graph_builder.add_edge(START, 'retrieve')
graph_builder.add_edge('retrieve', 'augmentation')
graph_builder.add_edge('augmentation', 'generation')
graph_builder.add_edge('generation', END)